# KNN para depressão (NHIS 2025): com e sem variáveis de ansiedade

Este notebook repete com o **KNN** a análise feita com o Random Forest (`../Random Forest/06_random_forest_depressao_2025.ipynb`): comparar o modelo com e sem variáveis de ansiedade e testar se o resultado depende das variáveis cognitivas e de solidão.

## Hipótese

> É possível prever depressão a partir de dados de saúde física e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

Esperamos desempenho **pior** sem os sintomas mentais. A hipótese se sustenta se o modelo ainda tiver poder preditivo real, bem acima do acaso.

## Cenários

1. **Cenário A (exploratório, com ansiedade):** removemos de $X$ apenas os sintomas de depressão (PHQ) e mantemos as variáveis de ansiedade (GAD-7).
2. **Cenário B (hipótese, sem sintomas mentais):** removemos também as variáveis de ansiedade.

## Roteiro

1. Preparação dos dados e cenários.
2. Função de treino e avaliação.
3. Cenários A e B e comparativo.
4. Ajuste de hiperparâmetros (`RandomizedSearchCV`) no cenário B.
5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?
6. Priorizando o recall: rastreio em saúde.
7. O que mais melhora o recall no KNN: ponderar as colunas.
8. Interpretação e conclusão.

## Desenho do experimento

- Mesma base, mesmas linhas (registros sem `-1` nas variáveis do cenário A), mesmo alvo binário (`0` = sem depressão, `1` = com depressão) e mesmo split treino/teste (80/20 estratificado, `random_state=42`) do Random Forest. Os resultados dos dois modelos são, portanto, comparáveis. A justificativa do alvo binário está no notebook do Random Forest.
- **Particularidades do KNN:**
  - O KNN **não aceita `PESO_AMOSTRAL` no treino**. O peso é usado apenas nas métricas, como no Random Forest.
  - O KNN **não tem `class_weight`**. Com o limiar padrão de 0,5, ele quase não detecta depressão (recall de cerca de 17% no cenário B), porque só 21% dos registros são positivos. Por isso classificamos como "com depressão" quando a probabilidade for **maior ou igual à prevalência do treino** (cerca de 21%), o equivalente a balancear as classes. O ROC-AUC não depende do limiar.
  - O KNN usa distância, então as variáveis categóricas passam por One-Hot e a razão renda/pobreza é padronizada. Não há importância de variáveis embutida; a influência de cada grupo é medida pelo teste de sensibilidade (seção 5).
- **Por que priorizar o recall:** em saúde, os dois erros não custam o mesmo. Um falso negativo é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um falso positivo é uma pessoa sem depressão que passa por uma avaliação a mais. Em um rastreio, é preferível encontrar mais casos, mesmo com mais falsos positivos. Por isso as seções 6 e 7 mostram quanto recall é possível ganhar e quanto isso custa em falsos positivos.

# 1. Preparação dos dados e definição dos cenários

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errada (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
nome_a, nome_b = CENARIOS
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS[nome_a])
df_modelo = df[(X_cenario_a != -1).all(axis=1).to_numpy()].copy()

# Mesmo split nos dois cenários (por posição)
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)
y_tr, y_te = df_modelo[TARGET_BIN].iloc[pos_tr], df_modelo[TARGET_BIN].iloc[pos_te]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]
prevalencia = np.average(y_te, weights=w_te)

# O KNN n?o tem class_weight: o limiar de decis?o ? a preval?ncia do treino (em vez de 0,5), equivalente a balancear as classes
LIMIAR = y_tr.mean()

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'Cenário {nome}: {n_feat} variáveis preditoras')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
Cenário A: com ansiedade (exploratório): 58 variáveis preditoras
Cenário B: sem sintomas mentais (hipótese): 46 variáveis preditoras


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, então a **única diferença** entre eles é o conjunto de variáveis. A configuração-base do KNN é `k = 15` vizinhos com voto ponderado pela distância. As métricas da classe positiva (com depressão) são ponderadas pelo `PESO_AMOSTRAL`, e a matriz de confusão é em **contagem de registros reais** do teste, sem peso. A função `avaliar` recebe o limiar de decisão (padrão: a prevalência do treino), o que permite, na seção 6, priorizar o recall.

In [10]:
def preparar_matrizes(colunas_remover):
    """Pré-processamento: padroniza a variável numérica e aplica One-Hot nas categóricas."""
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]
    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    return X.shape[1], preprocessor.fit_transform(X_tr), preprocessor.transform(X_te)


def avaliar(y_proba, limiar=LIMIAR):
    y_pred = (y_proba >= limiar).astype(int)
    metricas = {
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    return metricas, confusion_matrix(y_te, y_pred)


def treinar_avaliar(colunas_remover, **parametros):
    n_vars, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    config = dict(n_neighbors=15, weights='distance', n_jobs=-1)
    config.update(parametros)
    modelo = KNeighborsClassifier(**config).fit(X_tr_p, y_tr)
    metricas, matriz = avaliar(modelo.predict_proba(X_te_p)[:, 1])
    return {'Nº de variáveis': n_vars, **metricas}, matriz

resultados = {}

# 3. Cenários A e B

**Cenário A (exploratório):** mantém as variáveis de ansiedade. Mostra o desempenho quando há informação de saúde mental disponível.

In [11]:
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.8325
Acurácia: 0.8522
Precisão (depressão): 0.7394
Recall (depressão): 0.4891
F1 (depressão): 0.5888

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3310  172]
 [ 474  461]]


**Cenário B (hipótese):** sem sintomas de depressão **nem** de ansiedade. Sobram variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

In [12]:
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.7675
Acurácia: 0.8021
Precisão (depressão): 0.5467
Recall (depressão): 0.4969
F1 (depressão): 0.5206

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3082  400]
 [ 451  484]]


## Comparativo entre os cenários

A coluna **Variação (B − A)** mostra o que se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência.

In [13]:
tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.8325,0.7675,-0.0649,0.5000
Acurácia,0.8522,0.8021,-0.0502,NaN
Precisão (depressão),0.7394,0.5467,-0.1927,0.2163
Recall (depressão),0.4891,0.4969,0.0078,NaN
F1 (depressão),0.5888,0.5206,-0.0682,NaN


### Resultado (`k = 15`)

- O ROC-AUC cai de **0,833** (A) para **0,768** (B), uma perda de 0,065, e a precisão cai de 73,9% para 54,7%. O recall fica praticamente igual (cerca de 49%), porque o limiar foi ajustado à prevalência.
- Mesmo sem nenhum sintoma de saúde mental, o cenário B fica **0,27 de ROC-AUC acima do acaso** e acerta 54,7% dos casos apontados como depressão, **2,5 vezes a prevalência** (21,6%).
- Estes números usam `k = 15`, uma escolha fixa. O ajuste da seção 4 mostra que um `k` maior melhora bastante o resultado.

# 4. Ajuste de hiperparâmetros (cenário B)

Buscamos `n_neighbors`, o tipo de voto (`weights`) e a distância (`metric`) com `RandomizedSearchCV` (15 combinações, validação cruzada de 3 partes), **só no cenário B**, que testa a hipótese. A seleção usa o ROC-AUC e **só os dados de treino**; o teste não participa da escolha. Como o KNN não aceita peso amostral, a busca não é ponderada.

In [14]:
n_vars_b, X_tr_b, X_te_b = preparar_matrizes(CENARIOS[nome_b])

ESPACO_BUSCA = {
    'n_neighbors': [5, 11, 15, 21, 31, 51, 71, 101, 151],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
}

# O KNN não aceita PESO_AMOSTRAL no treino; a busca otimiza o ROC-AUC (não ponderado) em 3 partes do treino
busca = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1), ESPACO_BUSCA,
    n_iter=15, scoring='roc_auc', cv=3, random_state=RANDOM_STATE, n_jobs=1,
)
busca.fit(X_tr_b, y_tr)
print('Melhores hiperparâmetros:', busca.best_params_)

modelo_aj = busca.best_estimator_
metricas_aj, matriz_aj = avaliar(modelo_aj.predict_proba(X_te_b)[:, 1])
print('\nMatriz de confusão do modelo ajustado (nº de registros do teste):')
print(matriz_aj)
comparacao = pd.DataFrame({'B base (k=15)': {k: resultados[nome_b][0][k] for k in metricas_aj}, 'B ajustado': metricas_aj})
comparacao['Variação'] = comparacao['B ajustado'] - comparacao['B base (k=15)']
comparacao.round(4)

Melhores hiperparâmetros: {'weights': 'distance', 'n_neighbors': 151, 'metric': 'manhattan'}

Matriz de confusão do modelo ajustado (nº de registros do teste):
[[3167  315]
 [ 476  459]]


,B base (k=15),B ajustado,Variação
ROC-AUC,0.7675,0.8159,0.0484
Acurácia,0.8021,0.8187,0.0166
Precisão (depressão),0.5467,0.6079,0.0612
Recall (depressão),0.4969,0.4558,-0.0411
F1 (depressão),0.5206,0.5210,0.0004


### Resultado do ajuste

- A busca escolheu `k = 151` vizinhos, distância Manhattan e voto ponderado pela distância. O ROC-AUC sobe de 0,768 para **0,816**, a precisão de 54,7% para 60,8% e o F1 fica igual (0,52). O recall cai um pouco (de 49,7% para 45,6%).
- O ganho vem de usar **muitos mais vizinhos**: com poucos vizinhos o KNN se confunde com o ruído local. `k = 151` é o maior valor da grade, então um `k` ainda maior poderia melhorar um pouco mais; o ganho esperado, porém, é pequeno.
- Mesmo ajustado, o KNN fica abaixo do Random Forest no cenário B (ROC-AUC de 0,816 contra 0,841). É esperado: o Random Forest escolhe as variáveis informativas, enquanto o KNN trata todas as colunas One-Hot com o mesmo peso na distância. A seção 7 testa dar mais peso às colunas informativas.

# 5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?

`TIPO_DIFICULDADE_COGNITIVA` vem da pergunta sobre dificuldade para **lembrar ou se concentrar**, e dificuldade de concentração também é item do PHQ, que já removemos. Há o risco de essas variáveis reintroduzirem sintomas depressivos no modelo. O mesmo cuidado vale para a solidão. Repetimos o cenário B (mesmas linhas, mesmo split e os hiperparâmetros ajustados) retirando progressivamente:

1. `TIPO_DIFICULDADE_COGNITIVA` sozinha;
2. o **bloco cognitivo** inteiro (`TIPO_DIFICULDADE_COGNITIVA`, `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA`, que vêm da mesma pergunta);
3. o bloco cognitivo **e** `FREQ_SOLITARIO`.

Retirar só uma variável é um teste fraco, porque as outras do bloco carregam a mesma informação. Por isso o passo 2 é o mais informativo.

In [15]:
BLOCO_COGNITIVO = ['TIPO_DIFICULDADE_COGNITIVA', 'QTD_DIFICULDADE_COGNITIVA', 'FREQUENCIA_FALHA_MEMORIA']
SOLIDAO = ['FREQ_SOLITARIO']
COLUNAS_B = CENARIOS[nome_b]

SENSIBILIDADE = {
    'B (referência)': COLUNAS_B,
    'B sem TIPO_DIFICULDADE_COGNITIVA': COLUNAS_B + BLOCO_COGNITIVO[:1],
    'B sem bloco cognitivo': COLUNAS_B + BLOCO_COGNITIVO,
    'B sem bloco cognitivo e sem solidão': COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO,
}

res_sens = {nome: treinar_avaliar(remover, **busca.best_params_)[0] for nome, remover in SENSIBILIDADE.items()}

tabela_sens = pd.DataFrame(res_sens).T
tabela_sens['Variação do ROC-AUC vs. B'] = tabela_sens['ROC-AUC'] - tabela_sens.loc['B (referência)', 'ROC-AUC']
tabela_sens['Nº de variáveis'] = tabela_sens['Nº de variáveis'].astype(int)
print(f'Referência de acaso: ROC-AUC 0,5 e precisão {prevalencia:.1%} (prevalência)\n')
tabela_sens.round(4)

Referência de acaso: ROC-AUC 0,5 e precisão 21.6% (prevalência)



,Nº de variáveis,ROC-AUC,Acurácia,Precisão (depressão),Recall (depressão),F1 (depressão),Variação do ROC-AUC vs. B
B (referência),46,0.8159,0.8187,0.6079,0.4558,0.5210,0.0000
B sem TIPO_DIFICULDADE_COGNITIVA,45,0.8156,0.8196,0.6231,0.4199,0.5017,-0.0003
B sem bloco cognitivo,43,0.7989,0.8032,0.5550,0.4539,0.4994,-0.0171
B sem bloco cognitivo e sem solidão,42,0.7784,0.7856,0.5048,0.4722,0.4879,-0.0375


### Resultado do teste de sensibilidade

| Cenário | Variáveis | ROC-AUC | Precisão | Recall | F1 |
|---|---|---|---|---|---|
| B (referência) | 46 | 0,816 | 60,8% | 45,6% | 0,521 |
| sem `TIPO_DIFICULDADE_COGNITIVA` | 45 | 0,816 | 62,3% | 42,0% | 0,502 |
| sem bloco cognitivo | 43 | 0,799 | 55,5% | 45,4% | 0,499 |
| sem bloco cognitivo e sem solidão | 42 | **0,778** | 50,5% | 47,2% | 0,488 |

- **Sem o `TIPO`:** o ROC-AUC não muda (0,816). As outras variáveis do bloco assumem o papel dela.
- **Sem o bloco cognitivo:** o ROC-AUC cai 0,017. O bloco não é o que sustenta o resultado.
- **Sem o bloco e sem a solidão:** o ROC-AUC é 0,778 e a precisão 50,5%, **2,3 vezes a prevalência** (21,6%). O modelo continua bem acima do acaso.
- **Mesmo padrão do Random Forest:** lá o ROC-AUC foi de 0,841 para 0,804 nos mesmos passos. Dois algoritmos diferentes chegam à mesma conclusão.
- **Limite do teste:** ele retira só as variáveis que identificamos como suspeitas. Outras, como suporte social, também podem se relacionar com sintomas. O teste mostra robustez, não ausência total de sobreposição.

# 6. Priorizando o recall: rastreio em saúde

Em um rastreio de depressão, os dois erros não custam o mesmo. Um **falso negativo** é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um **falso positivo** é uma pessoa sem depressão que passa por uma avaliação a mais. Por isso é preferível encontrar mais casos, mesmo que isso piore os falsos positivos. O KNN, com o limiar na prevalência do treino (cerca de 21%), encontra menos da metade dos casos (recall de 45,6% no cenário B). Esta seção mostra quanto recall dá para ganhar **baixando o limiar** de probabilidade e quanto isso custa: o modelo sinaliza mais pessoas, o recall sobe e a precisão cai.

**Como o limiar é escolhido, sem olhar o teste:** calculamos a probabilidade prevista dos registros de treino com validação cruzada de 5 partes (`cross_val_predict`) e, para cada **recall-alvo** (80%, 85%, 90% e 95%), usamos o limiar de probabilidade que alcança esse recall entre os casos de depressão do treino. Depois aplicamos o limiar ao teste e medimos o que realmente aconteceu. Por isso o recall obtido no teste pode ficar um pouco acima ou abaixo do alvo.

**Como ler a tabela:** `Recall` e `Precisão` são ponderados pelo `PESO_AMOSTRAL`, como nas seções anteriores. `TP`, `FN`, `FP` e `TN` são **contagens de registros** do teste, sem peso, e `% da amostra sinalizada` e `Sinalizados por caso encontrado` também usam contagens. Este último é quantas pessoas precisam ser avaliadas para achar um caso de depressão.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

ALVOS_RECALL = [0.80, 0.85, 0.90, 0.95]


def analisar_limiares(colunas_remover, modelo):
    """Escolhe o limiar no treino (validação cruzada) para cada recall-alvo e mede o resultado no teste."""
    _, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    proba_te = clone(modelo).fit(X_tr_p, y_tr).predict_proba(X_te_p)[:, 1]
    proba_cv = cross_val_predict(clone(modelo), X_tr_p, y_tr, cv=5, method='predict_proba')[:, 1]
    proba_positivos = proba_cv[(y_tr == 1).to_numpy()]

    linhas, matrizes = {}, {}
    for alvo in [None] + ALVOS_RECALL:
        limiar = LIMIAR if alvo is None else np.quantile(proba_positivos, 1 - alvo)
        nome = 'Padrão (limiar = prevalência)' if alvo is None else f'Recall-alvo {alvo:.0%}'
        metricas, matrizes[nome] = avaliar(proba_te, limiar)
        tn, fp, fn, tp = matrizes[nome].ravel()
        linhas[nome] = {
            'Limiar': limiar,
            'Recall': metricas['Recall (depressão)'],
            'Precisão': metricas['Precisão (depressão)'],
            'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
            '% da amostra sinalizada': (tp + fp) / len(y_te),
            'Sinalizados por caso encontrado': (tp + fp) / tp,
        }
    return pd.DataFrame.from_dict(linhas, orient='index'), matrizes

In [ ]:
knn_ajustado = KNeighborsClassifier(**busca.best_params_, n_jobs=-1)
tabela_lim_b, matrizes_lim_b = analisar_limiares(COLUNAS_B, knn_ajustado)

print(f'Cenário B (KNN ajustado). Prevalência de depressão no teste (ponderada): {prevalencia:.1%}')
print('\nMatriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):')
print(matrizes_lim_b['Recall-alvo 90%'])
tabela_lim_b.round(3)

### Resultado (cenário B)

- **Mais recall, mais falsos positivos.** Do limiar padrão (a prevalência, 0,21) para o recall-alvo de 90%, o recall vai de 45,6% para **87,2%** e os casos perdidos caem de 476 para **107**. O custo são os falsos positivos, que passam de 315 para **1.679** (mais de 5 vezes), e a precisão, que cai de 60,8% para 34,0%.
- **Custo em avaliações.** Para cada caso encontrado, são avaliadas 1,7 pessoas no limiar padrão, 3,0 no recall-alvo de 90% e 3,5 no de 95%.
- **Continua melhor que o acaso, mas é o mais caro dos modelos.** No recall-alvo de 90%, o modelo sinaliza 56,8% da amostra e encontra 828 dos 935 casos. Um sorteio que sinalizasse a mesma fração encontraria cerca de 530. A precisão de 34,0% é 1,6 vez a prevalência (21,6%) e, no alvo de 95%, 1,4 vez (29,2%). Para o mesmo recall, o KNN gera mais falsos positivos que o Random Forest, o SVM e o XGBoost (1.679 contra 1.341 a 1.445).
- **O recall obtido no teste fica abaixo do alvo.** Ele ficou entre 0,4 e 4,3 pontos abaixo dos alvos (76,3%, 80,7%, 87,2% e 94,6%), porque o limiar vem do treino.
- **Qual alvo usar é uma decisão de saúde, não estatística.** Depende do custo de avaliar cada pessoa sinalizada e da capacidade do serviço. Se a prioridade é não deixar casos para trás, o alvo de 90% é um ponto de partida razoável: cerca de 3 em cada 10 pessoas sinalizadas têm depressão, e 107 dos 935 casos (11%) ficam sem identificação. A seção 7 mostra como reduzir os falsos positivos desse custo.

### Mesmo teste no cenário mais conservador

Repetimos a análise sem o bloco cognitivo e sem a solidão (seção 5), as variáveis mais próximas de sintomas depressivos.

In [ ]:
tabela_lim_cons, _ = analisar_limiares(COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO, knn_ajustado)
tabela_lim_cons.round(3)

- No limiar padrão, o cenário conservador encontra 469 dos 935 casos (recall de 47,2%, precisão de 50,5%). Para chegar a um recall próximo de 90% (**89,4%** no teste), o modelo sinaliza 63,6% da amostra, com 1.974 falsos positivos e precisão de **30,5%** (1,4 vez a prevalência).
- Portanto, **dá para atingir um recall alto mesmo sem as variáveis cognitivas e de solidão**, mas o custo é maior: 3,4 pessoas avaliadas por caso encontrado, contra 3,0 no cenário B completo.

# 7. O que mais melhora o recall no KNN: ponderar as colunas

Na seção 4, o KNN ficou abaixo do Random Forest porque trata todas as colunas One-Hot com o mesmo peso na distância: as colunas sem informação pesam tanto quanto as que importam. Diferente dos outros modelos, aqui **mudar o modelo pode ajudar de verdade**, e não só mover o ponto de operação.

Para separar os dois efeitos, usamos duas medidas que **não dependem do limiar**: o **PR-AUC** e a **precisão com recall de 90%**, que é a maior precisão que o modelo alcança quando exigimos que ele encontre pelo menos 90% dos casos do teste. Se uma variante aumenta o recall no limiar padrão, mas deixa essas duas medidas iguais, ela só moveu o ponto de operação. Um modelo sem informação teria PR-AUC e precisão iguais à prevalência (21,6%).

Variantes testadas no cenário B:

1. **O KNN ajustado da seção 4** (referência).
2. **Mais vizinhos** (`k = 301`, voto uniforme): a ideia mais simples, já que o `k` escolhido na seção 4 foi o maior da grade.
3. **Colunas ponderadas pela importância:** cada coluna é multiplicada pelo valor absoluto do seu coeficiente em um **SVM linear com penalidade L1** (o mesmo tipo de modelo do notebook do SVM), calculado **só nos dados de treino**. As colunas mais informativas passam a pesar mais na distância e as que o SVM zera saem dela. A força da penalidade (`C`) e o `k` são escolhidos por validação cruzada no treino, pelo PR-AUC.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


def precisao_no_recall(escore, recall_alvo=0.90):
    """Maior precisão (ponderada) que o modelo alcança no teste exigindo ao menos `recall_alvo` de recall."""
    precisao, recall, _ = precision_recall_curve(y_te, escore, sample_weight=w_te)
    return precisao[recall >= recall_alvo].max()


def resumir(escore, limiar=LIMIAR):
    """Medidas que não dependem do limiar e o resultado no limiar padrão."""
    metricas, matriz = avaliar(escore, limiar)
    tn, fp, fn, tp = matriz.ravel()
    return {
        'ROC-AUC': metricas['ROC-AUC'],
        'PR-AUC': average_precision_score(y_te, escore, sample_weight=w_te),
        'Precisão com recall de 90%': precisao_no_recall(escore),
        'Recall': metricas['Recall (depressão)'],
        'Precisão': metricas['Precisão (depressão)'],
        'Casos perdidos (FN)': fn,
        'Falsos positivos (FP)': fp,
    }


class PonderarColunas(BaseEstimator, TransformerMixin):
    """Multiplica cada coluna pelo |coeficiente| de um SVM linear L1 (calculado só no treino).

    As colunas mais informativas passam a pesar mais na distância do KNN e as que o SVM zera saem dela."""

    def __init__(self, C=0.02):
        self.C = C

    def fit(self, X, y):
        svm = LinearSVC(C=self.C, penalty='l1', dual=False, class_weight='balanced',
                        max_iter=20000, random_state=RANDOM_STATE)
        coeficientes = np.abs(svm.fit(X, y).coef_[0])
        self.pesos_ = coeficientes / coeficientes.max()
        return self

    def transform(self, X):
        return X * self.pesos_


busca_pond = GridSearchCV(
    make_pipeline(PonderarColunas(), KNeighborsClassifier(metric='manhattan')),
    {'ponderarcolunas__C': [0.005, 0.01, 0.02, 0.05], 'kneighborsclassifier__n_neighbors': [51, 101, 151, 251]},
    scoring='average_precision', cv=3, n_jobs=-1,
)
busca_pond.fit(X_tr_b, y_tr)
modelo_pond = busca_pond.best_estimator_
print('Melhores hiperparâmetros:', busca_pond.best_params_)
print('Colunas com peso maior que zero:', int((modelo_pond[0].pesos_ > 0).sum()), 'de', X_tr_b.shape[1])

mais_vizinhos = KNeighborsClassifier(n_neighbors=301, metric='manhattan', n_jobs=-1).fit(X_tr_b, y_tr)
escores = {
    'Ajustado na seção 4 (k = 151, voto por distância)': modelo_aj.predict_proba(X_te_b)[:, 1],
    'Mais vizinhos (k = 301, voto uniforme)': mais_vizinhos.predict_proba(X_te_b)[:, 1],
    'Colunas ponderadas pelo SVM L1': modelo_pond.predict_proba(X_te_b)[:, 1],
}
tabela_alavancas = pd.DataFrame.from_dict({nome: resumir(e) for nome, e in escores.items()}, orient='index')
print(f'\nReferência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência ({prevalencia:.1%})\n')
tabela_alavancas.round(4)

### Resultado

- **Mais vizinhos quase não muda nada.** Com `k = 301` o ROC-AUC vai de 0,816 para 0,821 e o PR-AUC fica igual (0,596 contra 0,600), e o recall no limiar padrão cai de 45,6% para 42,9%. O `k = 151` da seção 4 não era o gargalo.
- **Ponderar as colunas melhora o modelo de verdade.** A busca escolheu `C = 0,02` e `k = 101`, e 102 das 193 colunas ficaram com peso maior que zero (as outras 91 saem da distância). O ROC-AUC sobe de 0,816 para **0,843** e o PR-AUC de 0,600 para **0,646**, ganhos bem acima da variação esperada de um único teste (erro-padrão de cerca de 0,01 no ROC-AUC). A precisão com recall de 90% sobe de 0,319 para 0,335, um ganho menor, dentro dessa variação.
- **O recall no limiar padrão sobe de 45,6% para 66,8%**, e os casos perdidos caem de 476 para 287. O custo são os falsos positivos (315 para 631) e a precisão (60,8% para 52,9%). Como a curva também sobe, esse ganho não é só deslocar o ponto de operação.
- **O KNN passa a empatar com o Random Forest** em ROC-AUC (0,843 contra 0,841), mas continua abaixo do SVM e do XGBoost (0,856).

In [ ]:
# O KNN com colunas ponderadas no limiar escolhido pelo recall-alvo (mesma análise da seção 6)
tabela_lim_pond, matrizes_lim_pond = analisar_limiares(COLUNAS_B, modelo_pond)
print('Matriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):')
print(matrizes_lim_pond['Recall-alvo 90%'])
print()
display(tabela_lim_pond.round(3))

# Teste de sensibilidade (seção 5) com o KNN de colunas ponderadas
res_sens_pond = {}
for nome, remover in SENSIBILIDADE.items():
    _, X_tr_p, X_te_p = preparar_matrizes(remover)
    metricas, _ = avaliar(clone(modelo_pond).fit(X_tr_p, y_tr).predict_proba(X_te_p)[:, 1])
    res_sens_pond[nome] = metricas
pd.DataFrame(res_sens_pond).T.round(4)

### Resultado: o KNN de colunas ponderadas no recall-alvo e na sensibilidade

- No recall-alvo de 90%, o KNN ponderado encontra 840 dos 935 casos (recall de 88,0%), com 1.530 falsos positivos e precisão de 37,0%. O KNN da seção 6 precisava de 1.679 falsos positivos (precisão de 34,0%) para um recall de 87,2%: são **149 falsos positivos a menos (9%)** com um recall um pouco maior.
- Ainda assim, o KNN gera mais falsos positivos que o Random Forest (1.445), o XGBoost (1.393) e o SVM (1.341) no mesmo alvo.
- **Teste de sensibilidade:** o ROC-AUC do KNN ponderado é 0,843 no cenário B, 0,825 sem o bloco cognitivo e **0,796** sem o bloco e sem a solidão (0,778 no KNN da seção 5). O padrão é o mesmo dos outros modelos: a queda total é de 0,047, e o modelo continua bem acima do acaso.

# 8. Interpretação e conclusão

## Conclusão

**ROC-AUC (não depende do limiar):**

| Modelo | ROC-AUC (A) | ROC-AUC (B) | B sem bloco cognitivo e sem solidão |
|---|---|---|---|
| XGBoost ajustado | 0,915 (base) | 0,856 | 0,811 |
| SVM linear ajustado | 0,915 (base) | 0,856 | 0,813 |
| KNN com colunas ponderadas | - | 0,843 | 0,796 |
| Random Forest | 0,915 | 0,841 | 0,804 |
| KNN ajustado (seção 4) | 0,833 (base, `k = 15`) | 0,816 | 0,778 |

**Pontos de operação no cenário B** (modelos ajustados). Recall e precisão são ponderados pelo `PESO_AMOSTRAL` e os falsos positivos são contagens do teste. Cada modelo tem um limiar padrão diferente. No recall-alvo de 90%, o limiar vem do treino (seção 6 deste notebook e seções equivalentes dos outros):

| Modelo | Limiar padrão: recall e precisão | Recall-alvo de 90%: recall obtido | Precisão | Falsos positivos | % da amostra sinalizada |
|---|---|---|---|---|---|
| SVM linear | 72,4% e 52,7% | 86,9% | 38,5% | 1.341 | 48,9% |
| XGBoost | 73,2% e 52,1% | 88,4% | 38,8% | 1.393 | 50,5% |
| Random Forest | 59,7% e 57,7% | 88,3% | 38,0% | 1.445 | 51,7% |
| KNN com colunas ponderadas | 66,8% e 52,9% | 88,0% | 37,0% | 1.530 | 53,7% |
| KNN ajustado (seção 4) | 45,6% e 60,8% | 87,2% | 34,0% | 1.679 | 56,8% |

**O que mais melhora o recall em cada algoritmo** (seção 7 deste notebook e seções equivalentes dos outros):

| Algoritmo | Alavanca testada | Recall no limiar padrão | Melhora a curva de precisão × recall? |
|---|---|---|---|
| Random Forest | peso 6 para a classe positiva | 59,7% → 71,5% | Não: só desloca o ponto de operação |
| KNN | colunas ponderadas pelo SVM L1 | 45,6% → 66,8% | **Sim:** PR-AUC 0,600 → 0,646 e ROC-AUC 0,816 → 0,843 |
| SVM linear | peso 5 e 8 para a classe positiva | 72,4% → 78,8% e 84,7% | Não |
| XGBoost | `scale_pos_weight` × 2 | 73,2% → 85,5% | Não |

- **A hipótese se sustenta com o KNN.** Sem nenhum sintoma de saúde mental, o KNN fica 0,32 de ROC-AUC acima do acaso (0,816) e, com as colunas ponderadas, 0,34 (0,843), com precisão de 53% a 61% contra 21,6% de prevalência. Dados físicos, funcionais e de contexto social carregam informação real sobre depressão.
- **Como esperado, é pior que com ansiedade.** A perda é de cerca de 0,065 de ROC-AUC com `k = 15`.
- **O resultado é robusto.** Retirar as variáveis mais próximas de sintomas depressivos (cognição e solidão) reduz o ROC-AUC em 0,038 no KNN ajustado e em 0,047 no KNN de colunas ponderadas, e o modelo continua bem acima do acaso.
- **Ponderar as colunas é o que mais melhora o KNN.** O ROC-AUC sobe de 0,816 para 0,843, o PR-AUC de 0,600 para 0,646, e o recall no limiar padrão de 45,6% para 66,8%. Com isso o KNN empata com o Random Forest no cenário B (0,843 contra 0,841) e fica um pouco abaixo no cenário conservador (0,796 contra 0,804). Continua abaixo de SVM e XGBoost (0,856). Entre os quatro algoritmos, é o único em que mudar o modelo ajudou, e não só mover o ponto de operação.
- **Para o rastreio, o KNN é o mais caro dos quatro.** No recall-alvo de 90%, o KNN ajustado sinaliza 56,8% da amostra, com 1.679 falsos positivos e precisão de 34,0%. Com as colunas ponderadas, são 53,7%, 1.530 falsos positivos e 37,0%. SVM, XGBoost e Random Forest precisam de 1.341 a 1.445 falsos positivos para o mesmo recall.
- **O limiar pesa muito.** Com o limiar na prevalência, o KNN ajustado encontra só 45,6% dos casos. Com o recall-alvo de 90%, encontra 87,2%, ao custo de 1.679 falsos positivos (315 no limiar padrão). Mesmo no cenário conservador (sem cognição e sem solidão), um recall de 90% é alcançável, com precisão de 30,5%.

## Limitações

- O KNN não usa `PESO_AMOSTRAL` no treino, só nas métricas, e o limiar de decisão padrão foi ajustado à prevalência para compensar a falta de `class_weight`.
- O recall-alvo (80% a 95%) é uma decisão de saúde, não estatística. O notebook mostra o custo de cada alvo, mas não define qual usar. O limiar é escolhido no treino, e o recall obtido no teste varia alguns pontos em torno do alvo.
- A ponderação das colunas usa os coeficientes de um SVM linear L1 treinado só nos dados de treino, com `C` e `k` escolhidos por validação cruzada. Outras formas de escolher os pesos podem dar resultados um pouco diferentes.
- Um único split treino/teste. A busca de hiperparâmetros usa validação cruzada, mas a comparação final usa um só teste, com cerca de 900 casos positivos. Diferenças de um a dois pontos entre variantes ficam dentro dessa variação.
- Registros completos apenas (22.084 de 24.215).
- Os modelos mostram **associação preditiva**, não causalidade.